# Groupby, Pivot, Merge

In [1]:
import pandas as pd 
import numpy as np 

univ = ['AAPL','MSFT','BAC','GS']
dates = pd.date_range('20110101','20201231')
df = pd.DataFrame(np.random.randn(len(dates),len(univ)),index=dates,columns=univ)
df

,AAPL,MSFT,BAC,GS
2011-01-01,0.127555,0.989822,-0.366338,0.971959
2011-01-02,-0.560884,-0.124174,0.272694,-0.272722
2011-01-03,0.336284,0.699852,1.836202,-0.661373
2011-01-04,-1.053897,0.328808,1.552531,1.547947
2011-01-05,1.556168,-1.937461,0.991215,-1.176823
...,...,...,...,...
2020-12-27,0.977667,-0.840796,-0.300323,0.833040
2020-12-28,-0.206261,-0.665929,0.865881,0.784767
2020-12-29,0.959073,0.464488,-0.244648,0.063460
2020-12-30,-0.517104,1.240429,-0.700744,-0.227458


In [2]:
sector = {'AAPL':'Tech','MSFT':'Tech','BAC':'Fin','GS':'Fin'}
data=[]
for dt in dates:
    for x in univ:
        data.append([x,sector[x],dt,np.random.randn(),np.random.randn()])
data = pd.DataFrame(data,columns=['ticker','sector','date','signal1','signal2'])
data

,ticker,sector,date,signal1,signal2
0,AAPL,Tech,2011-01-01,2.117530,-0.035326
1,MSFT,Tech,2011-01-01,-0.935777,-1.075920
2,BAC,Fin,2011-01-01,1.447426,-1.013461
3,GS,Fin,2011-01-01,0.358617,0.667986
4,AAPL,Tech,2011-01-02,-0.504650,-0.403448
...,...,...,...,...,...
14607,GS,Fin,2020-12-30,-0.318061,-0.555858
14608,AAPL,Tech,2020-12-31,0.159108,-0.609334
14609,MSFT,Tech,2020-12-31,0.120104,0.583847
14610,BAC,Fin,2020-12-31,-0.330351,2.219011


## Groupby

In [11]:
# whats the average signal of each ticker?
### data.groupby('ticker').mean()
data

,ticker,sector,date,signal1,signal2
0,AAPL,Tech,2011-01-01,2.117530,-0.035326
1,MSFT,Tech,2011-01-01,-0.935777,-1.075920
2,BAC,Fin,2011-01-01,1.447426,-1.013461
3,GS,Fin,2011-01-01,0.358617,0.667986
4,AAPL,Tech,2011-01-02,-0.504650,-0.403448
...,...,...,...,...,...
14607,GS,Fin,2020-12-30,-0.318061,-0.555858
14608,AAPL,Tech,2020-12-31,0.159108,-0.609334
14609,MSFT,Tech,2020-12-31,0.120104,0.583847
14610,BAC,Fin,2020-12-31,-0.330351,2.219011


In [13]:
data.groupby('ticker').mean(numeric_only=True)

,signal1,signal2
ticker,,
AAPL,0.006178,0.048693
BAC,-0.008882,-0.036994
GS,0.023808,-0.019539
MSFT,0.000235,-0.008783


In [14]:
# whats the average signal of each sector on each day?
data.groupby(['sector', 'date']).mean(numeric_only=True)

signal1   signal2
sector date                          
Fin    2011-01-01  0.903022 -0.172737
       2011-01-02  1.072052 -0.781100
       2011-01-03 -0.873587 -0.006639
       2011-01-04  0.694268 -1.057314
       2011-01-05  0.397080  1.082549
...                     ...       ...
Tech   2020-12-27  0.482330  0.221704
       2020-12-28 -1.155488  0.006217
       2020-12-29 -0.056227 -0.799852
       2020-12-30 -0.792726 -0.251186
       2020-12-31  0.139606 -0.012743

[7306 rows x 2 columns]

In [15]:
# can index before applying the function
data.groupby(['sector','date'])[['signal1']].mean()

signal1
sector date                
Fin    2011-01-01  0.903022
       2011-01-02  1.072052
       2011-01-03 -0.873587
       2011-01-04  0.694268
       2011-01-05  0.397080
...                     ...
Tech   2020-12-27  0.482330
       2020-12-28 -1.155488
       2020-12-29 -0.056227
       2020-12-30 -0.792726
       2020-12-31  0.139606

[7306 rows x 1 columns]

In [19]:
# use apply with groupby to pass an arbitrary function
def max_minus_min(x):
    return x.max()-x.min()

data.groupby(['sector','date'])[['signal1','signal2']].apply(max_minus_min)

signal1   signal2
sector date                          
Fin    2011-01-01  1.088809  1.681447
       2011-01-02  0.978840  1.198542
       2011-01-03  0.235223  0.134453
       2011-01-04  3.654260  1.266591
       2011-01-05  1.373867  1.955090
...                     ...       ...
Tech   2020-12-27  0.806203  2.627341
       2020-12-28  1.670334  0.764660
       2020-12-29  2.502164  1.040172
       2020-12-30  0.241570  0.097016
       2020-12-31  0.039004  1.193181

[7306 rows x 2 columns]

In [20]:
# return a dataframe instead of a series 
def demean(x):
    return x - x.mean()

data.groupby(['sector','date'])[['signal1','signal2']].apply(demean)

signal1   signal2
sector date                                
Fin    2011-01-01 2      0.544404 -0.840723
                  3     -0.544404  0.840723
       2011-01-02 6      0.489420  0.599271
                  7     -0.489420 -0.599271
       2011-01-03 10    -0.117612  0.067226
...                           ...       ...
Tech   2020-12-29 14601  1.251082 -0.520086
       2020-12-30 14604 -0.120785 -0.048508
                  14605  0.120785  0.048508
       2020-12-31 14608  0.019502 -0.596590
                  14609 -0.019502  0.596590

[14612 rows x 2 columns]

In [21]:
df = data.groupby(['sector','date'])[['signal1','signal2']].mean()
df

signal1   signal2
sector date                          
Fin    2011-01-01  0.903022 -0.172737
       2011-01-02  1.072052 -0.781100
       2011-01-03 -0.873587 -0.006639
       2011-01-04  0.694268 -1.057314
       2011-01-05  0.397080  1.082549
...                     ...       ...
Tech   2020-12-27  0.482330  0.221704
       2020-12-28 -1.155488  0.006217
       2020-12-29 -0.056227 -0.799852
       2020-12-30 -0.792726 -0.251186
       2020-12-31  0.139606 -0.012743

[7306 rows x 2 columns]

In [22]:
df.groupby(level=0).mean()

,signal1,signal2
sector,,
Fin,0.007463,-0.028267
Tech,0.003206,0.019955


In [23]:
month = [x.month for x in data['date']]

In [28]:
### data.groupby(month).mean()
data.groupby(month).mean(numeric_only=True)

,signal1,signal2
1,0.038890,0.006030
2,-0.009016,0.023858
3,0.102323,0.001686
4,-0.051543,0.001159
5,0.031072,-0.081225
6,0.004559,-0.041464
7,0.043068,-0.027251
8,-0.082049,-0.010790
9,0.009120,0.024159
10,0.029166,-0.021256


In [29]:
for key,val in data.groupby('sector'):
    print (key)

Fin
Tech


In [30]:
# iterating in a groupby
for key,val in data.groupby('sector'):
    print (val)

      ticker sector       date   signal1   signal2
2        BAC    Fin 2011-01-01  1.447426 -1.013461
3         GS    Fin 2011-01-01  0.358617  0.667986
6        BAC    Fin 2011-01-02  1.561472 -0.181829
7         GS    Fin 2011-01-02  0.582632 -1.380371
10       BAC    Fin 2011-01-03 -0.991199  0.060588
...      ...    ...        ...       ...       ...
14603     GS    Fin 2020-12-29  0.523213  1.170794
14606    BAC    Fin 2020-12-30  0.216268 -0.377992
14607     GS    Fin 2020-12-30 -0.318061 -0.555858
14610    BAC    Fin 2020-12-31 -0.330351  2.219011
14611     GS    Fin 2020-12-31 -0.464910 -0.529268

[7306 rows x 5 columns]
      ticker sector       date   signal1   signal2
0       AAPL   Tech 2011-01-01  2.117530 -0.035326
1       MSFT   Tech 2011-01-01 -0.935777 -1.075920
4       AAPL   Tech 2011-01-02 -0.504650 -0.403448
5       MSFT   Tech 2011-01-02  0.125516  0.459324
8       AAPL   Tech 2011-01-03 -0.006409 -1.140304
...      ...    ...        ...       ...       ...
14601 

## Pivot

In [31]:
data

,ticker,sector,date,signal1,signal2
0,AAPL,Tech,2011-01-01,2.117530,-0.035326
1,MSFT,Tech,2011-01-01,-0.935777,-1.075920
2,BAC,Fin,2011-01-01,1.447426,-1.013461
3,GS,Fin,2011-01-01,0.358617,0.667986
4,AAPL,Tech,2011-01-02,-0.504650,-0.403448
...,...,...,...,...,...
14607,GS,Fin,2020-12-30,-0.318061,-0.555858
14608,AAPL,Tech,2020-12-31,0.159108,-0.609334
14609,MSFT,Tech,2020-12-31,0.120104,0.583847
14610,BAC,Fin,2020-12-31,-0.330351,2.219011


In [32]:
# put Signal1 into a df with columns tickers and rows dates
df1 = data.set_index(['date','ticker'])['signal1'].unstack(level=1)
df1

ticker,AAPL,BAC,GS,MSFT
date,,,,
2011-01-01,2.117530,1.447426,0.358617,-0.935777
2011-01-02,-0.504650,1.561472,0.582632,0.125516
2011-01-03,-0.006409,-0.991199,-0.755976,0.572472
2011-01-04,0.082161,2.521398,-1.132862,-0.829932
2011-01-05,-0.465370,-0.289853,1.084013,0.857615
...,...,...,...,...
2020-12-27,0.079228,0.790740,-1.141878,0.885432
2020-12-28,-0.320321,-0.224435,1.579687,-1.990655
2020-12-29,-1.307309,0.808862,0.523213,1.194855


In [33]:
df2 = data.pivot_table(index='date',columns='ticker',values='signal1')
df2

ticker,AAPL,BAC,GS,MSFT
date,,,,
2011-01-01,2.117530,1.447426,0.358617,-0.935777
2011-01-02,-0.504650,1.561472,0.582632,0.125516
2011-01-03,-0.006409,-0.991199,-0.755976,0.572472
2011-01-04,0.082161,2.521398,-1.132862,-0.829932
2011-01-05,-0.465370,-0.289853,1.084013,0.857615
...,...,...,...,...
2020-12-27,0.079228,0.790740,-1.141878,0.885432
2020-12-28,-0.320321,-0.224435,1.579687,-1.990655
2020-12-29,-1.307309,0.808862,0.523213,1.194855


In [34]:
df1.equals(df2)

True

In [36]:
### data.pivot_table(index='date',columns='sector',values='signal1',aggfunc=np.mean)
data.pivot_table(index='date',columns='sector',values='signal1',aggfunc='mean')

sector,Fin,Tech
date,,
2011-01-01,0.903022,0.590876
2011-01-02,1.072052,-0.189567
2011-01-03,-0.873587,0.283031
2011-01-04,0.694268,-0.373886
2011-01-05,0.397080,0.196123
...,...,...
2020-12-27,-0.175569,0.482330
2020-12-28,0.677626,-1.155488
2020-12-29,0.666037,-0.056227


In [38]:
### data.pivot_table(index='date',columns='sector',values='signal1',aggfunc=np.max)
data.pivot_table(index='date',columns='sector',values='signal1',aggfunc='max')

sector,Fin,Tech
date,,
2011-01-01,1.447426,2.117530
2011-01-02,1.561472,0.125516
2011-01-03,-0.755976,0.572472
2011-01-04,2.521398,0.082161
2011-01-05,1.084013,0.857615
...,...,...
2020-12-27,0.790740,0.885432
2020-12-28,1.579687,-0.320321
2020-12-29,0.808862,1.194855


## Merge
Check out documentation to learn more: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html

In [39]:
data2=[]
for dt in dates:
    for x in ['Tech','Fin']:
        data2.append([x,dt,np.random.randn(),np.random.randn()])
        
data2 = pd.DataFrame(data2,columns=['sector','date','signal3','signal4'])
data2

,sector,date,signal3,signal4
0,Tech,2011-01-01,2.508095,-0.108039
1,Fin,2011-01-01,-0.241539,-0.598506
2,Tech,2011-01-02,1.014726,-0.641696
3,Fin,2011-01-02,-0.876055,-0.807538
4,Tech,2011-01-03,-2.204307,-0.993463
...,...,...,...,...
7301,Fin,2020-12-29,0.188511,0.299235
7302,Tech,2020-12-30,0.532125,1.206276
7303,Fin,2020-12-30,-0.772393,0.041560
7304,Tech,2020-12-31,0.631552,-0.926221


In [40]:
data.merge(data2,left_on=['sector','date'],right_on=['sector','date'])

,ticker,sector,date,signal1,signal2,signal3,signal4
0,AAPL,Tech,2011-01-01,2.117530,-0.035326,2.508095,-0.108039
1,MSFT,Tech,2011-01-01,-0.935777,-1.075920,2.508095,-0.108039
2,BAC,Fin,2011-01-01,1.447426,-1.013461,-0.241539,-0.598506
3,GS,Fin,2011-01-01,0.358617,0.667986,-0.241539,-0.598506
4,AAPL,Tech,2011-01-02,-0.504650,-0.403448,1.014726,-0.641696
...,...,...,...,...,...,...,...
14607,GS,Fin,2020-12-30,-0.318061,-0.555858,-0.772393,0.041560
14608,AAPL,Tech,2020-12-31,0.159108,-0.609334,0.631552,-0.926221
14609,MSFT,Tech,2020-12-31,0.120104,0.583847,0.631552,-0.926221
14610,BAC,Fin,2020-12-31,-0.330351,2.219011,-2.422867,0.867578
